In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd

import os

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt
from pyearthtools.pipeline.operations.xarray.join import GeospatialTimeSeriesMerge
import site_archive_nci

In [ ]:
# Currently only includes GHI, need solar elevation too!!!
himawari = petdata.archive.Himawari(
    [
    'surface_global_irradiance',
    'solar_elevation'
    ]
)

variables_of_interest = [
    'huss',
    'hus850',
    'hus700',
    'hus500',
    'psl',
    'tas',
    'ta850',
    'ta700',
    'ta500',
    # 'rsds',
]
frequency = '1hr'
domain_id = 'AUST-04'
barra = petdata.archive.BARRA_V2(
    variables_of_interest,
    frequency=frequency,
    domain_id=domain_id
)

In [ ]:
# TO DO:
# WRITE CUSTOM FUNCTION FOR PREPROCESSING RAW BANDS INTO GRIDDED LAT/LON
band8 = petdata.archive.HimawariChannels(bands=['OBS_B08'])

In [ ]:
satpipe = petpipe.Pipeline(
    himawari,
    petdata.transform.region.Select(latitude=-33.84,longitude=151.22, method='nearest'),
    iterator=petpipe.iterators.DateRange('20240101T00', '20240201T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

barpipe = petpipe.Pipeline(
    barra,
    petdata.transform.region.Select(latitude=-33.84,longitude=151.22, method='nearest'),
    petdata.transform.region.ISelect(height=0),
    iterator=petpipe.iterators.DateRange('20240101T00', '20240201T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

ds_him = satpipe['2024-01']
ds_bar = barpipe['2024-01']

In [ ]:
dfs = []

# make DFs for pressure level vars
for p in ds_bar.pressure.values:
    tmp = ds_bar.sel(pressure=p).to_dataframe()
    tmp = tmp.rename(columns={'hus': f'hus{int(p)}'})
    tmp = tmp.rename(columns={'ta': f'ta{int(p)}'})
    dfs.append(tmp[['hus' + str(int(p))]])
    dfs.append(tmp[['ta' + str(int(p))]])

# Add surface level vars
base = ds_bar[['huss', 'psl', 'tas']].to_dataframe()

df_bar = base.join(dfs)
df_bar = df_bar.drop(columns=['latitude', 'longitude', 'crs', 'height'], errors='ignore')

In [ ]:
df_him = ds_him[['surface_global_irradiance', 'solar_elevation']].to_dataframe()
# df_him = ds_him[['surface_global_irradiance']].to_dataframe()
df_him = df_him.drop(columns=['latitude', 'longitude'], errors='ignore')

data = df_bar.join(df_him, how='inner')

for n in range(1, 11):
    data[f'surface_global_irradiance_t{n}'] = data['surface_global_irradiance'].shift(-n)

In [ ]:
data.to_csv('/scratch/er8/cd3022/xgb_datasets/test_data.csv')